# Bernini-R 指令式视频/图像编辑 — Colab G4 启动脚本

**模型**：字节跳动 Bernini-R（基于 Wan2.2 的统一编辑模型）　**运行时**：G4 高 RAM（RTX PRO 6000 Blackwell / 96GB）

支持六种任务：`img` 图像编辑 / `v2v` 视频风格化 / `rv2v` 参考引导视频编辑 / `r2v` 参考图生视频 / `ads2v` 内容插入 / `t2v` 文生视频。

全程用 ComfyUI **原生节点**，**不需要任何自定义节点**。

| 项 | 值 |
| --- | --- |
| 模型数量 | 4 个（主模型 / 文本编码器 / VAE / 加速LoRA） |
| 内网穿透 | FRP tcp 无 token，端口 **8092** |
| 访问地址 | http://usoren.usdream.dpdns.org:8092 |

按顺序跑 Cell 1 → 5。重启界面只需重跑 Cell 4 与 Cell 5。端口与 SCAIL-2 笔记本（8091）错开，两个会话可同时跑。

> **Cell 1 会强制把 ComfyUI 重置到最新 master，并升级前端包与模板库 pip 包**。`comfyui-frontend-package` 和 `comfyui-workflow-templates` 是独立 pip 包，**不会随 git pull 更新**——模板库搜不到 Bernini-R、节点显示缺失，基本都是这个原因。

> **服务端前提**：frps 配置里不能有 `auth.token`，改完 `systemctl restart frps`；8092 需空闲且在 `allowPorts` 内。

> **防断连**：在 Colab 页面按 F12 → Console 粘贴：`setInterval(()=>{document.body.click()},60000)`


In [ ]:
# ==========================================
# Cell 1: 安装 / 强制更新 ComfyUI 到最新 nightly
# ------------------------------------------
# 重要: Bernini-R 的原生节点、以及模板库里的 Bernini-R 条目,
# 分属三个不同的来源:
#   1. 节点代码  -> ComfyUI git 仓库 (comfy_extras/)
#   2. 前端界面  -> pip 包 comfyui-frontend-package
#   3. 模板库    -> pip 包 comfyui-workflow-templates
# 只做 git pull 不会更新后两者, 这就是“模板库搜不到”的原因。
# ==========================================
import os
import subprocess

REPO = "/content/ComfyUI"
print("=== 🚀 安装 / 更新 ComfyUI ===")
%cd /content

if not os.path.exists(REPO):
    !git clone https://github.com/comfyanonymous/ComfyUI

# git pull 遇到本地改动会失败且不报错, 改用 fetch + reset --hard 强制对齐远端
!cd {REPO} && git fetch --all --quiet && git reset --hard origin/master --quiet
!cd {REPO} && git log -1 --format="当前 ComfyUI 版本: %h  %ad  %s" --date=short

%cd /content/ComfyUI
!pip install -q -r requirements.txt huggingface_hub hf_transfer

# 关键一步: 前端包 / 模板库 / 内置文档 都是独立 pip 包, 必须单独升级
print("\n=== ⬆️ 升级前端包与模板库 ===")
!pip install -q -U comfyui-frontend-package comfyui-workflow-templates comfyui-embedded-docs

# ComfyUI-Manager 只作排错用, 本工作流不需要任何自定义节点
MGR = "/content/ComfyUI/custom_nodes/ComfyUI-Manager"
if not os.path.exists(MGR):
    !git clone -q https://github.com/ltdrdata/ComfyUI-Manager.git {MGR}

# ---------------- 自检 ----------------
print("\n=== 🔍 环境自检 ===")

import torch
print("torch:", torch.__version__, "| cuda:", torch.version.cuda)
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability(0)
    print("GPU:", torch.cuda.get_device_name(0), "| capability:", cap)
    if cap[0] >= 12:
        print("✅ Blackwell, fp16 主模型可直接跑, 无需量化")
else:
    print("⚠️ 未检测到 GPU, 请检查运行时类型")

import importlib.metadata as md
for pkg in ["comfyui-frontend-package", "comfyui-workflow-templates"]:
    try:
        print(pkg + ": " + md.version(pkg))
    except Exception:
        print(pkg + ": 未安装")

# 1) 原生节点是否存在
node_hit = subprocess.run(
    "grep -ril bernini /content/ComfyUI/comfy_extras/ | head -5",
    shell=True, capture_output=True, text=True).stdout.strip()
if node_hit:
    print("\n✅ 找到 Bernini 原生节点:\n" + node_hit)
else:
    print("\n❌ 未找到 Bernini 节点")
    print("   说明上游 master 还没合入该节点, 或 reset 失败。")
    print("   处理: !rm -rf /content/ComfyUI 后重跑本格")

# 2) 模板库里是否已含 Bernini 模板
tpl = subprocess.run(
    "ls /usr/local/lib/python3*/dist-packages/comfyui_workflow_templates/templates/ "
    "2>/dev/null | grep -i bernini",
    shell=True, capture_output=True, text=True).stdout.strip()
if tpl:
    print("✅ 模板库已含 Bernini-R 条目:\n" + tpl)
else:
    print("⚠️ 模板库里没有 Bernini-R, 不影响使用 — Cell 3 会直接从 GitHub 下载工作流 JSON")

print("\n✅ Cell 1 完成")


In [ ]:
# ==========================================
# Cell 2: 下载 Bernini-R 所需的 4 个模型
# 清单来自 ComfyUI 官方文档 docs.comfy.org/tutorials/video/bytedance/bernini-r
# ==========================================
import os
import shutil
from huggingface_hub import hf_hub_download
from concurrent.futures import ThreadPoolExecutor

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    print("⚠️ 未读到 HF_TOKEN (左侧 🔑 Secrets), 公开仓库仍可下载")

M = "/content/ComfyUI/models"

downloads = [
    # --- 主模型 (fp16, 96GB 显存直接上满血版) ---
    {"repo": "Comfy-Org/Bernini-R",
     "file": "wan2.2_bernini_r_fp16.safetensors",
     "alt": ["diffusion_models/wan2.2_bernini_r_fp16.safetensors",
             "diffusion_models/wan2.2_bernini_r_high_noise_fp16.safetensors"],
     "dir": M + "/diffusion_models"},

    # --- 文本编码器 ---
    {"repo": "Comfy-Org/Wan_2.1_ComfyUI_repackaged",
     "file": "split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors",
     "dir": M + "/text_encoders"},

    # --- VAE ---
    {"repo": "Kijai/WanVideo_comfy",
     "file": "Wan2_1_VAE_bf16.safetensors",
     "dir": M + "/vae"},

    # --- 加速 LoRA (注意是 T2V 版, 与 SCAIL-2 的 I2V 版不同) ---
    {"repo": "Kijai/WanVideo_comfy",
     "file": "lightx2v_T2V_14B_cfg_step_distill_v2_lora_rank64_bf16.safetensors",
     "alt": ["Lightx2v/lightx2v_T2V_14B_cfg_step_distill_v2_lora_rank64_bf16.safetensors"],
     "dir": M + "/loras"},

    # ==========================================
    # 以下为可选项, 默认不下载
    # ==========================================
    # ❌ fp8 / int8 / mxfp8 量化版: 给 12-24GB 小显存卡用的, G4 上用 fp16 画质更好
    # {"repo": "Comfy-Org/Bernini-R", "file": "diffusion_models/wan2.2_bernini_r_low_noise_fp8_scaled.safetensors", "dir": M + "/diffusion_models"},
    # ❌ 1.3B 轻量版 (~2.6GB), 只适合低显存试跑
    # {"repo": "Comfy-Org/Bernini-R", "file": "wan2.1_bernini_1.3B_fp16.safetensors", "dir": M + "/diffusion_models"},
]


def fetch(task):
    os.makedirs(task["dir"], exist_ok=True)
    candidates = [task["file"]] + task.get("alt", [])
    name = os.path.basename(task["file"])
    final = os.path.join(task["dir"], name)
    last = None

    if os.path.exists(final):
        print("⏩ 已存在, 跳过: " + name)
        return

    for cand in candidates:
        try:
            p = hf_hub_download(repo_id=task["repo"], filename=cand, local_dir=task["dir"])
            if os.path.abspath(p) != os.path.abspath(final):
                shutil.move(p, final)
            print("✅ 完成: " + name)
            return
        except Exception as e:
            last = e
    print("❌ 失败 " + name + ": " + str(last))


print("🚀 开始下载 " + str(len(downloads)) + " 个模型 (主模型约 32GB, 耗时较长)...")
with ThreadPoolExecutor(max_workers=4) as ex:
    list(ex.map(fetch, downloads))

print("\n=== 📁 模型目录清单 ===")
for sub in ["diffusion_models", "text_encoders", "vae", "loras"]:
    d = os.path.join(M, sub)
    if os.path.isdir(d):
        for f in os.listdir(d):
            fp = os.path.join(d, f)
            if os.path.isfile(fp):
                print("  " + sub + "/" + f + "  (" + str(round(os.path.getsize(fp) / 1e9, 2)) + " GB)")

print("\n🎉 模型就绪")


In [ ]:
# ==========================================
# Cell 3: 下载官方工作流模板
# 不依赖模板库 pip 包, 直接拉 GitHub 上的最新 JSON
# 读取位置: ComfyUI 左侧“工作流”面板 (不是“模板”面板)
# ==========================================
import os
import urllib.request

wf_dir = "/content/ComfyUI/user/default/workflows"
os.makedirs(wf_dir, exist_ok=True)

BASE = ("https://raw.githubusercontent.com/Comfy-Org/workflow_templates/"
        "main/templates/")

templates = [
    "video_bernini_r_image_editing.json",
    "video_bernini_r_video_editing.json",
]

for t in templates:
    try:
        urllib.request.urlretrieve(BASE + t, os.path.join(wf_dir, t))
        print("✅ 已下载: " + t)
    except Exception as e:
        print("❌ 失败 " + t + ": " + str(e))

print("\n📁 当前 workflows 目录:")
for f in sorted(os.listdir(wf_dir)):
    print("  " + f)

print("\n👉 启动后点左侧边栏的「工作流」面板 (文件夹图标) 打开, 不是「模板」面板")


In [ ]:
# ==========================================
# Cell 4: FRP 内网穿透配置 (tcp, 无 token)
# ==========================================
import os
import subprocess

FRP_HOST    = "usoren.usdream.dpdns.org"
FRP_PORT    = 7000
REMOTE_PORT = 8092
FRP_VER     = "0.56.0"
FRP_DIR     = "/content/frp_" + FRP_VER + "_linux_amd64"
ACCESS_URL  = "http://" + FRP_HOST + ":" + str(REMOTE_PORT)

if not os.path.exists(FRP_DIR + "/frpc"):
    print("⏳ 下载 frp ...")
    subprocess.run(
        "wget -qO- https://github.com/fatedier/frp/releases/download/v"
        + FRP_VER + "/frp_" + FRP_VER + "_linux_amd64.tar.gz | tar -xz -C /content",
        shell=True)
assert os.path.exists(FRP_DIR + "/frpc"), "frpc 下载失败, 请重跑本格"
subprocess.run("chmod +x " + FRP_DIR + "/frpc", shell=True)

frpc_conf = (
    'serverAddr = "' + FRP_HOST + '"\n'
    'serverPort = ' + str(FRP_PORT) + '\n'
    'loginFailExit = false\n'
    'transport.tcpMux = true\n'
    'transport.poolCount = 5\n'
    'log.to = "/content/frpc.log"\n'
    'log.level = "info"\n\n'
    '[[proxies]]\n'
    'name = "bernini_colab"\n'
    'type = "tcp"\n'
    'localIP = "127.0.0.1"\n'
    'localPort = 8188\n'
    'remotePort = ' + str(REMOTE_PORT) + '\n')

with open(FRP_DIR + "/frpc.toml", "w") as f:
    f.write(frpc_conf)

print("✅ frpc.toml 已写入:")
print("-" * 50)
print(frpc_conf)
print("-" * 50)
print("👉 启动后访问: " + ACCESS_URL)
print("⚠️  地址必须带端口, 不带端口看到的是 frps 自带的 404 页")


In [ ]:
# ==========================================
# Cell 5: 启动 frpc + ComfyUI (重启界面只跑这一格)
# ==========================================
import os
import time
import threading
import subprocess
import configparser

COMFY      = "/content/ComfyUI"
FRP_DIR    = "/content/frp_0.56.0_linux_amd64"
ACCESS_URL = "http://usoren.usdream.dpdns.org:8092"

assert os.path.isdir(COMFY), "找不到 ComfyUI, 请先跑 Cell 1"
assert os.path.exists(FRP_DIR + "/frpc.toml"), "找不到 frpc.toml, 请先跑 Cell 4"
os.environ["OPENCV_IO_ENABLE_OPENEXR"] = "1"


def keep_alive():
    while True:
        time.sleep(300)
        print("\n[Keep-Alive] 保持连接活跃...")


threading.Thread(target=keep_alive, daemon=True).start()

for log in ["/content/comfy.log", "/content/frpc.log"]:
    if os.path.exists(log):
        os.remove(log)

# --- 1. 先起 frpc (秒级) ---
print("⏳ 启动 FRP 穿透...")
subprocess.run("pkill -f '" + FRP_DIR + "/frpc' || true", shell=True)
subprocess.Popen(FRP_DIR + "/frpc -c " + FRP_DIR +
                 "/frpc.toml >> /content/frpc.log 2>&1", shell=True)
time.sleep(6)

frp_log = ""
if os.path.exists("/content/frpc.log"):
    frp_log = open("/content/frpc.log", errors="ignore").read()

if "start proxy success" in frp_log:
    print("✅ FRP 隧道已建立 -> " + ACCESS_URL)
elif "token in login" in frp_log:
    print("❌ 服务端开了 token 验证, 请删掉 frps 的 auth.token 后 systemctl restart frps")
elif "already used" in frp_log:
    print("❌ 远程端口被占用, 请改 Cell 4 的 REMOTE_PORT")
elif "port not allowed" in frp_log:
    print("❌ 端口不在 frps 的 allowPorts 范围内")
else:
    print("⚠️ frpc 日志:")
print(frp_log[-1200:] if frp_log else "(日志为空)")

# --- 2. 关掉 Manager 启动时的联网 Fetch ---
for cfg in [COMFY + "/user/__manager/config.ini",
            COMFY + "/user/default/ComfyUI-Manager/config.ini",
            COMFY + "/custom_nodes/ComfyUI-Manager/config.ini"]:
    os.makedirs(os.path.dirname(cfg), exist_ok=True)
    c = configparser.ConfigParser()
    if os.path.exists(cfg):
        c.read(cfg)
    if "default" not in c:
        c["default"] = {}
    c["default"]["network_mode"] = "private"
    with open(cfg, "w") as f:
        c.write(f)

# --- 3. 启动 ComfyUI ---
LAUNCH = ("python main.py --listen 127.0.0.1 --port 8188 "
          "--enable-cors-header '*' --preview-method auto")
print("\n⏳ 启动 ComfyUI...")
subprocess.Popen(LAUNCH + " > /content/comfy.log 2>&1", shell=True, cwd=COMFY)

ready = False
for i in range(120):
    time.sleep(2)
    if not os.path.exists("/content/comfy.log"):
        continue
    txt = open("/content/comfy.log", errors="ignore").read()
    if "To see the GUI go to" in txt:
        ready = True
        print("✅ ComfyUI ready")
        break
    if "Traceback" in txt and i > 10:
        print("❌ 启动报错:")
        print(txt[-3000:])
        break

if not ready:
    print("--- 日志尾部 ---")
    if os.path.exists("/content/comfy.log"):
        print(open("/content/comfy.log", errors="ignore").read()[-3000:])

print("\n============================================================")
print("🎉 ComfyUI : " + ACCESS_URL)
print("⚠️  地址必须带 :8092")
print("============================================================\n")

subprocess.run("tail -f /content/comfy.log", shell=True)
